In [25]:
import pandas as pd
import numpy as np
import joblib
import category_encoders as ce
import optuna
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [26]:
df = pd.read_csv('datasets/kingametric_credit_risk.csv')

In [27]:
df.shape

(8744, 45)

In [28]:
df.columns.to_list()

['Annual_Income',
 'Monthly_Inhand_Salary',
 'Num_Bank_Accounts',
 'Num_Credit_Card',
 'Interest_Rate',
 'Num_of_Loan',
 'Delay_from_due_date',
 'Num_of_Delayed_Payment',
 'Changed_Credit_Limit',
 'Num_Credit_Inquiries',
 'Credit_Mix',
 'Outstanding_Debt',
 'Credit_Utilization_Ratio',
 'Credit_History_Age',
 'Payment_of_Min_Amount',
 'Total_EMI_per_month',
 'Amount_invested_monthly',
 'Payment_Behaviour',
 'Monthly_Balance',
 'normalized_dti',
 'normalized_emi',
 'normalized_delinquency',
 'normalized_credit_history',
 'normalized_savings',
 'normalized_utilization',
 'normalized_utilization_risk',
 'normalized_inquiry_intensity',
 'normalized_investment_ratio',
 'normalized_loan_burden_index',
 'behavioral_risk_indicator',
 'credit_mix_quality',
 'normalized_savings_capacity_ratio',
 'population_density_factor',
 'Default_Flag',
 'Borrower_Tier',
 'Debt_Stress_Index',
 'Repayment_Stress',
 'Credit_Exposure',
 'Liquidity_Buffer',
 'Behavioral_Risk_Composite',
 'Credit_Instability',
 'P

In [29]:
df["Debt_Stress"] = df["normalized_dti"] * df["normalized_utilization"]
df["Repayment_Stress"] = df["normalized_emi"] * df["normalized_delinquency"]
df["Liquidity_Index"] = df["normalized_savings"] * df["normalized_emi"]
df["Credit_Exposure"] = df["Num_Credit_Card"] * df["Credit_Utilization_Ratio"]
df["Risk_Index"] = (df["normalized_dti"] + df["normalized_utilization"] + df["normalized_delinquency"]) / 3

df["Income_Delinq"] = df["Annual_Income"] * df["normalized_delinquency"]
df["Loan_DTI"] = df["Num_of_Loan"] * df["normalized_dti"]
df["Income_Q"] = pd.qcut(df["Annual_Income"], 4, labels=['Q1', 'Q2', 'Q3', 'Q4']).astype('str')

In [30]:
bad_features = ['Payment_Behaviour', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'behavioral_risk_indicator', 'normalized_savings']

df.drop(columns=bad_features, inplace=True)

In [31]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

In [32]:
df.shape

(8744, 46)

In [33]:
X = df.drop('Default_Flag', axis=1)
y = df['Default_Flag']

In [35]:
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("Categorical features:", categorical_features)

Categorical features: ['Credit_Mix', 'Payment_of_Min_Amount', 'Borrower_Tier', 'Income_Q']
